# AR 01 RNS Entity Screen

**Author:** Noah Green CPA CFE

This Cross-Border Diligence Atlas lab notebook demonstrates a source-literacy workflow for Argentina Registro Nacional de Sociedades (RNS)-shaped synthetic rows. It loads `data/synthetic/argentina_rns.csv`, applies the project parser, and exports a small redacted table for review.

This notebook does not query an official registry, does not provide a legal conclusion, and does not draw country-level risk conclusions.

In [1]:
from pathlib import Path
import csv
import sys

cwd = Path.cwd()
lab_root = cwd if (cwd / "src" / "crossborder_dd").exists() else cwd.parent
if not (lab_root / "src" / "crossborder_dd").exists():
    raise FileNotFoundError("Run this notebook from the lab root or notebooks directory.")

src_path = str(lab_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from crossborder_dd.argentina_rns_parser import parse_rns_row

data_path = lab_root / "data" / "synthetic" / "argentina_rns.csv"
output_path = lab_root / "data" / "redacted_outputs" / "AR_01_rns_entity_screen.csv"

data_path.relative_to(lab_root), output_path.relative_to(lab_root)

(PosixPath('data/synthetic/argentina_rns.csv'),
 PosixPath('data/redacted_outputs/AR_01_rns_entity_screen.csv'))

## Load Synthetic RNS Rows

The input is a synthetic fixture. It is shaped like a minimal RNS entity row, but it is not a republication of live registry data.

In [2]:
with data_path.open(newline="", encoding="utf-8") as handle:
    synthetic_rows = list(csv.DictReader(handle))

synthetic_rows

[{'cuit': '20123456786', 'razon_social': 'Sociedad Sintetica Argentina SA'},
 {'cuit': '00000000000', 'razon_social': 'Identificador Invalido SRL'}]

## Parse Rows With The Project Parser

The parsing logic lives in `src/crossborder_dd/argentina_rns_parser.py`. The notebook calls `parse_rns_row` so the demonstration stays aligned with the tested helper module.

In [3]:
parsed_rows = [parse_rns_row(row) for row in synthetic_rows]

parsed_rows

[{'country': 'Argentina',
  'identifier': '20123456786',
  'identifier_type': 'CUIT/CDI',
  'identifier_valid': True,
  'entity_name': 'Sociedad Sintetica Argentina SA',
  'entity_name_normalized': 'sociedad sintetica argentina sa'},
 {'country': 'Argentina',
  'identifier': '00000000000',
  'identifier_type': 'CUIT/CDI',
  'identifier_valid': False,
  'entity_name': 'Identificador Invalido SRL',
  'entity_name_normalized': 'identificador invalido srl'}]

## Build Redacted Source-Literacy Output

The output keeps only synthetic display names and masked identifiers. Each row carries its limitation and local-counsel escalation note so the table cannot be mistaken for a legal-status finding.

In [4]:
def mask_identifier(identifier):
    digits = str(identifier or "")
    if len(digits) <= 4:
        return "REDACTED"
    return f"{digits[:2]}*******{digits[-2:]}"


def lead_flag(parsed_row):
    if parsed_row["identifier_valid"]:
        return "format_passes_check_digit"
    return "format_fails_check_digit"


redacted_rows = []
for index, parsed in enumerate(parsed_rows, start=1):
    redacted_rows.append(
        {
            "sample_id": f"AR_RNS_SYN_{index:02d}",
            "country": parsed["country"],
            "source_name": "Registro Nacional de Sociedades synthetic fixture",
            "identifier_type": parsed["identifier_type"],
            "identifier_redacted": mask_identifier(parsed["identifier"]),
            "entity_name_redacted": f"SYNTHETIC_ENTITY_{index:02d}",
            "entity_name_normalized": parsed["entity_name_normalized"],
            "identifier_valid": parsed["identifier_valid"],
            "match_type": "identifier_format_check",
            "lead_flag": lead_flag(parsed),
            "limitation": "Synthetic row only; a format result or no-hit does not prove registration, good standing, ownership, authority, or adverse status.",
            "escalation_note": "Escalate to Argentina local counsel before interpreting legal status or using this result in deal terms.",
        }
    )

redacted_rows

[{'sample_id': 'AR_RNS_SYN_01',
  'country': 'Argentina',
  'source_name': 'Registro Nacional de Sociedades synthetic fixture',
  'identifier_type': 'CUIT/CDI',
  'identifier_redacted': '20*******86',
  'entity_name_redacted': 'SYNTHETIC_ENTITY_01',
  'entity_name_normalized': 'sociedad sintetica argentina sa',
  'identifier_valid': True,
  'match_type': 'identifier_format_check',
  'lead_flag': 'format_passes_check_digit',
  'limitation': 'Synthetic row only; a format result or no-hit does not prove registration, good standing, ownership, authority, or adverse status.',
  'escalation_note': 'Escalate to Argentina local counsel before interpreting legal status or using this result in deal terms.'},
 {'sample_id': 'AR_RNS_SYN_02',
  'country': 'Argentina',
  'source_name': 'Registro Nacional de Sociedades synthetic fixture',
  'identifier_type': 'CUIT/CDI',
  'identifier_redacted': '00*******00',
  'entity_name_redacted': 'SYNTHETIC_ENTITY_02',
  'entity_name_normalized': 'identificador

In [5]:
fieldnames = list(redacted_rows[0].keys())
with output_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames, lineterminator="\n")
    writer.writeheader()
    writer.writerows(redacted_rows)

output_path.relative_to(lab_root)

PosixPath('data/redacted_outputs/AR_01_rns_entity_screen.csv')

## What This Proves / What It Cannot Prove

**What this proves:** the lab can load a synthetic Argentina RNS-shaped CSV, call the shared `parse_rns_row` helper, normalize entity names and CUIT/CDI-style identifiers, flag invalid identifier format, and export a redacted source-literacy table with limitations preserved.

**What this cannot prove:** it does not prove that an entity exists in RNS, is active, is in good standing, has authority to transact, has particular owners or officers, or has any adverse or clean status. It does not convert a no-hit into a finding, does not make a legal conclusion, and does not create a jurisdiction-level score.

**Local-counsel escalation:** escalate to Argentina local counsel before interpreting registry status, relying on official-source results in closing conditions, or using the result for price, indemnity, escrow, termination, licensing, KYC/AML, procurement, tax, or corporate-authority decisions.